# 멀티모달 Cross-Attention Fusion
**Video × Text × Audio** 세 모달리티를 Cross-Attention으로 퓨전하여 감정 이진 분류(Positive/Negative) 수행

| 모달 | 파일 | 차원 |
|------|------|------|
| Video | `data/video_features_256.pkl` | 256 |
| Text  | `data/text_features_256(basic+earlystop).pkl` | 256 |
| Audio | `data/audio_feat_hubert_origin.pkl` | 256 |

> **pkl 파일 위치 안내**  
> - Text: sooyeon 브랜치의 `text_features_256(basic+earlystop).pkl` → `data/` 폴더에 복사  
> - Audio: jeein 브랜치의 `audio/audio_feat_hubert_origin.pkl` → `data/` 폴더에 복사

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

## 1. 피처 로드

In [ ]:
# ── pkl 경로 설정 ──────────────────────────────────────────────────────────────
VIDEO_PKL = 'data/video_features_256.pkl'
TEXT_PKL  = 'data/text_features_256(basic+earlystop).pkl'
AUDIO_PKL = 'data/audio_feat_hubert_origin.pkl'

def load_pkl(path):
    """(feats, labels) 튜플 또는 {'audio_feat':..,'labels':..} dict 모두 처리."""
    with open(path, 'rb') as f:
        data = pickle.load(f)
    if isinstance(data, dict):
        feats  = data['audio_feat']  # audio 포맷
        labels = data['labels']
    else:  # tuple / list
        feats, labels = data

    # numpy array로 통일
    if isinstance(feats, torch.Tensor):
        feats = feats.cpu().float().numpy()
    else:
        feats = np.array(feats, dtype=np.float32)

    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()
    else:
        labels = np.array(labels)

    return feats, labels

vid_feats,   vid_labels   = load_pkl(VIDEO_PKL)
text_feats,  text_labels  = load_pkl(TEXT_PKL)
audio_feats, audio_labels = load_pkl(AUDIO_PKL)

print(f'Video  feats: {vid_feats.shape}   labels: {vid_labels.shape}')
print(f'Text   feats: {text_feats.shape}  labels: {text_labels.shape}')
print(f'Audio  feats: {audio_feats.shape} labels: {audio_labels.shape}')

## 2. 샘플 정렬 확인 & 데이터셋 구성

In [ ]:
# 세 모달리티 N이 같아야 함
N_vid, N_txt, N_aud = len(vid_labels), len(text_labels), len(audio_labels)
print(f'샘플 수 → Video: {N_vid}  Text: {N_txt}  Audio: {N_aud}')

assert N_vid == N_txt == N_aud, (
    f'샘플 수 불일치! Video={N_vid}, Text={N_txt}, Audio={N_aud}\n'
    '※ 각 pkl은 전체 데이터셋 기준으로 저장된 피처여야 합니다.'
)

# 레이블 일치 확인
vid_lbl_int   = vid_labels.astype(int)
text_lbl_int  = text_labels.astype(int)
audio_lbl_int = audio_labels.astype(int)
label_match = (vid_lbl_int == text_lbl_int).all() and (vid_lbl_int == audio_lbl_int).all()
print(f'레이블 완전 일치: {label_match}')
if not label_match:
    print('  ※ 레이블 불일치 → video 레이블을 기준으로 사용합니다.')

labels = vid_lbl_int
print(f'Positive(1): {(labels==1).sum()}  Negative(0): {(labels==0).sum()}')

In [ ]:
# ── Train / Val / Test 분할 (8:1:1, stratified) ───────────────────────────────
idx = np.arange(len(labels))

tr_idx, te_idx = train_test_split(idx, test_size=0.2,  random_state=SEED, stratify=labels)
tr_idx, va_idx = train_test_split(tr_idx, test_size=0.125, random_state=SEED,
                                   stratify=labels[tr_idx])  # 0.125 × 0.8 = 0.1

print(f'Train: {len(tr_idx)}  Val: {len(va_idx)}  Test: {len(te_idx)}')

def to_tensor_split(idx_arr):
    v = torch.tensor(vid_feats[idx_arr],   dtype=torch.float32)
    t = torch.tensor(text_feats[idx_arr],  dtype=torch.float32)
    a = torch.tensor(audio_feats[idx_arr], dtype=torch.float32)
    y = torch.tensor(labels[idx_arr],      dtype=torch.long)
    return v, t, a, y

tr_v, tr_t, tr_a, tr_y = to_tensor_split(tr_idx)
va_v, va_t, va_a, va_y = to_tensor_split(va_idx)
te_v, te_t, te_a, te_y = to_tensor_split(te_idx)

BATCH = 32
train_loader = DataLoader(TensorDataset(tr_v, tr_t, tr_a, tr_y), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(va_v, va_t, va_a, va_y), batch_size=BATCH, shuffle=False)
test_loader  = DataLoader(TensorDataset(te_v, te_t, te_a, te_y), batch_size=BATCH, shuffle=False)

## 3. 모델 정의 — Cross-Attention Fusion

```
Video(256) ─┐
Text (256) ─┼─► CrossAttn(V→T,A) + CrossAttn(T→V,A) + CrossAttn(A→V,T) ─► concat(768) ─► MLP ─► 2
Audio(256) ─┘
```

각 모달리티가 나머지 두 모달리티에 대해 **Query → (Key, Value)** Cross-Attention을 수행합니다.

In [ ]:
class CrossAttention(nn.Module):
    """단일 Cross-Attention 블록.
    query: (B, D_q)  key/value: (B, N_kv, D_kv)
    out:   (B, D_q)
    """
    def __init__(self, d_query: int, d_kv: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=d_query, num_heads=num_heads,
            kdim=d_kv, vdim=d_kv,
            batch_first=True, dropout=dropout
        )
        self.norm = nn.LayerNorm(d_query)
        self.ff   = nn.Sequential(
            nn.Linear(d_query, d_query * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_query * 2, d_query),
        )
        self.norm2 = nn.LayerNorm(d_query)

    def forward(self, query, kv):
        """query: (B, D)  kv: (B, 2, D) — 두 모달리티를 sequence로 쌓아서 넣음"""
        q = query.unsqueeze(1)              # (B, 1, D)
        attn_out, _ = self.mha(q, kv, kv)  # (B, 1, D)
        attn_out = attn_out.squeeze(1)      # (B, D)
        x = self.norm(query + attn_out)     # residual
        x = self.norm2(x + self.ff(x))      # FF + residual
        return x


class TriModalCrossAttnFusion(nn.Module):
    """
    Video(v), Text(t), Audio(a) 세 모달리티를 Cross-Attention으로 퓨전.

    각 모달리티가 나머지 두를 attend:
      v' = CrossAttn(Q=v, KV=[t, a])
      t' = CrossAttn(Q=t, KV=[v, a])
      a' = CrossAttn(Q=a, KV=[v, t])

    최종: concat([v', t', a']) → MLP → logits
    """
    def __init__(self, feat_dim: int = 256, num_heads: int = 4,
                 dropout: float = 0.1, num_classes: int = 2):
        super().__init__()
        # 입력 정규화 (각 모달리티의 피처 스케일 통일)
        self.ln_v = nn.LayerNorm(feat_dim)
        self.ln_t = nn.LayerNorm(feat_dim)
        self.ln_a = nn.LayerNorm(feat_dim)

        # Cross-Attention: 각 모달리티 → 나머지 두 모달리티
        self.ca_v = CrossAttention(feat_dim, feat_dim, num_heads, dropout)  # v attends (t,a)
        self.ca_t = CrossAttention(feat_dim, feat_dim, num_heads, dropout)  # t attends (v,a)
        self.ca_a = CrossAttention(feat_dim, feat_dim, num_heads, dropout)  # a attends (v,t)

        # 분류 헤드
        fused_dim = feat_dim * 3  # 256 × 3 = 768
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, v, t, a):
        """v, t, a: (B, 256)"""
        v = self.ln_v(v)
        t = self.ln_t(t)
        a = self.ln_a(a)

        # KV: 두 모달리티를 sequence 차원으로 묶음 (B, 2, 256)
        kv_for_v = torch.stack([t, a], dim=1)  # video가 볼 context
        kv_for_t = torch.stack([v, a], dim=1)  # text가 볼 context
        kv_for_a = torch.stack([v, t], dim=1)  # audio가 볼 context

        v_out = self.ca_v(v, kv_for_v)  # (B, 256)
        t_out = self.ca_t(t, kv_for_t)  # (B, 256)
        a_out = self.ca_a(a, kv_for_a)  # (B, 256)

        fused  = torch.cat([v_out, t_out, a_out], dim=-1)  # (B, 768)
        logits = self.classifier(fused)                     # (B, 2)
        return logits


model = TriModalCrossAttnFusion(feat_dim=256, num_heads=4, dropout=0.2).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n학습 가능한 파라미터: {total_params:,}')

## 4. 학습

In [ ]:
# ── 학습 설정 ──────────────────────────────────────────────────────────────────
NUM_EPOCHS   = 50
LR           = 3e-4
PATIENCE     = 10    # Early Stopping patience
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# ── Early Stopping ──────────────────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best_acc  = 0.0
        self.best_state = None

    def step(self, val_acc, model):
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc  = val_acc
            self.counter   = 0
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

early_stop = EarlyStopping(patience=PATIENCE)

# ── 학습 루프 ──────────────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for v, t, a, y in loader:
            v, t, a, y = v.to(device), t.to(device), a.to(device), y.to(device)
            logits = model(v, t, a)
            loss   = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * y.size(0)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += y.size(0)
    return total_loss / total, correct / total

print(f'{'Epoch':>6} | {'Tr Loss':>8} | {'Tr Acc':>7} | {'Va Loss':>8} | {'Va Acc':>7}')
print('-' * 50)

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)

    stop = early_stop.step(va_acc, model)
    if epoch % 5 == 0 or stop:
        print(f'{epoch:>6} | {tr_loss:>8.4f} | {tr_acc:>6.4f} | {va_loss:>8.4f} | {va_acc:>6.4f}'
              + (' ← best' if early_stop.counter == 0 else ''))
    if stop:
        print(f'\nEarly Stopping @ epoch {epoch}  (best val_acc={early_stop.best_acc:.4f})')
        break

# 최적 가중치 복원
model.load_state_dict(early_stop.best_state)
print(f'\n최적 모델 복원 완료  (val_acc={early_stop.best_acc:.4f})')

## 5. 학습 곡선 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train', markersize=3)
axes[0].plot(epochs_range, history['val_loss'],   'r-s', label='Val',   markersize=3)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train', markersize=3)
axes[1].plot(epochs_range, history['val_acc'],   'r-s', label='Val',   markersize=3)
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1])

plt.suptitle('Cross-Attention Fusion — Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fusion_training_curves.png', dpi=150)
plt.show()
print('저장: fusion_training_curves.png')

## 6. 테스트 성능 평가

In [ ]:
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for v, t, a, y in test_loader:
        v, t, a = v.to(device), t.to(device), a.to(device)
        logits = model(v, t, a)
        preds  = logits.argmax(1).cpu().tolist()
        all_preds.extend(preds)
        all_targets.extend(y.tolist())

test_acc = accuracy_score(all_targets, all_preds)
test_f1  = f1_score(all_targets, all_preds, average='macro')

print(f'=== Test 결과 ===')
print(f'Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Macro F1 : {test_f1:.4f}')
print()
print(classification_report(all_targets, all_preds, target_names=['Negative(0)', 'Positive(1)']))

## 7. Confusion Matrix & 모달리티별 기여도 분석

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(all_targets, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Test Confusion Matrix  (acc={test_acc:.3f})')
plt.tight_layout()
plt.savefig('fusion_confusion_matrix.png', dpi=150)
plt.show()
print('저장: fusion_confusion_matrix.png')

In [ ]:
# ── 모달리티 ablation (단일 모달리티 제로-마스킹) ───────────────────────────────
def eval_with_mask(loader, mask_v=False, mask_t=False, mask_a=False):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for v, t, a, y in loader:
            v, t, a = v.to(device), t.to(device), a.to(device)
            if mask_v: v = torch.zeros_like(v)
            if mask_t: t = torch.zeros_like(t)
            if mask_a: a = torch.zeros_like(a)
            logits = model(v, t, a)
            preds.extend(logits.argmax(1).cpu().tolist())
            targets.extend(y.tolist())
    return accuracy_score(targets, preds)

ablation = {
    'All modalities (full)':       eval_with_mask(test_loader),
    'Text + Audio (Video masked)': eval_with_mask(test_loader, mask_v=True),
    'Video + Audio (Text masked)': eval_with_mask(test_loader, mask_t=True),
    'Video + Text (Audio masked)': eval_with_mask(test_loader, mask_a=True),
    'Audio only':                  eval_with_mask(test_loader, mask_v=True, mask_t=True),
    'Text only':                   eval_with_mask(test_loader, mask_v=True, mask_a=True),
    'Video only':                  eval_with_mask(test_loader, mask_t=True, mask_a=True),
}

print('=== Ablation Study (Test Accuracy) ===')
for name, acc in ablation.items():
    bar = '█' * int(acc * 30)
    print(f'  {name:<40} {acc:.4f}  {bar}')

# 시각화
fig, ax = plt.subplots(figsize=(9, 4))
names = list(ablation.keys())
accs  = list(ablation.values())
colors = ['steelblue' if i == 0 else 'salmon' for i in range(len(names))]
bars = ax.barh(names, accs, color=colors)
ax.set_xlim(0, 1)
ax.set_xlabel('Test Accuracy')
ax.set_title('Ablation Study — Modality Contribution')
for bar, acc in zip(bars, accs):
    ax.text(acc + 0.01, bar.get_y() + bar.get_height()/2,
            f'{acc:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('fusion_ablation.png', dpi=150)
plt.show()
print('저장: fusion_ablation.png')

## 8. 모델 저장

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'test_acc':  test_acc,
    'test_f1':   test_f1,
    'val_acc':   early_stop.best_acc,
    'config': {'feat_dim': 256, 'num_heads': 4, 'dropout': 0.2},
}, 'cross_attn_fusion_best.pt')

print('모델 저장 완료: cross_attn_fusion_best.pt')
print(f'\n최종 성능 요약')
print(f'  Val  Accuracy : {early_stop.best_acc:.4f}')
print(f'  Test Accuracy : {test_acc:.4f}')
print(f'  Test Macro F1 : {test_f1:.4f}')